# IdiomX v2 LLM Advanced Enrichment Pipeline Notebook

**Author:** Ayman Ali Sharara  
**Project:** IdiomX – Neural Understanding of English Idioms  
**Repository:** IdiomX Dataset and Enrichment Pipeline  
**Year:** 2026  

---

## Purpose of this Notebook

This notebook orchestrates the **IdiomX v2 advanced enrichment pipeline** in a clean, reproducible, research-oriented workflow.

The original IdiomX enrichment pipeline successfully produced a large bilingual idiom dataset with idiomatic and literal examples. However, analysis of the first version revealed several opportunities for improvement:

- limited structural diversity in generated examples
- repeated or templated sentence patterns across idioms
- absence of explicit hard negatives for retrieval-style evaluation
- no dedicated adversarial examples for robustness testing
- limited explanation/rationale fields for explainable AI use cases
- no explicit discourse-style coverage such as dialogue, formal, or social-media contexts

To address these limitations, **IdiomX v2** extends the enrichment process with:

- controlled context diversity
- balanced idiomatic and literal generation across discourse styles
- hard negative idioms
- semantic paraphrases of idiomatic meaning
- adversarial / borderline examples
- bilingual explanations at both idiom level and example level
- richer support for multi-task benchmarking

---

## Why a New Version?

This notebook starts from scratch because the enrichment schema, validation rules, and downstream dataset structure have changed significantly compared with the earlier pipeline.

Rather than patching the old notebook, this version provides a **clean and reproducible v2 workflow** that is easier to:

- review
- rerun
- debug
- publish
- share with future users of the IdiomX repository

---

## Final Output Goal

The final enriched IdiomX v2 dataset is saved in both:

- **CSV**
- **Parquet**

under the project folder:

- `data/final/`

This ensures that the final research-ready dataset is separate from:
- raw inputs
- temporary sample files
- intermediate batch files
- validation-stage files

---
---

## IdiomX v2 Dataset Schema

The final IdiomX v2 enriched dataset includes source metadata, idiom-level semantic fields, example-level bilingual annotations, robustness-oriented fields, and audit/validation fields.

### Core identity fields
- **idiom_id**: stable identifier for the idiom record
- **idiom_canonical**: normalized canonical idiom form
- **idiom_surface**: exact surface form used in the example when applicable

### Source / provenance fields
- **example**: original source example from the pre-enrichment dataset
- **source**: lexical source of the idiom
- **source_type**: type of source, such as dictionary
- **source_url**: original source URL if available
- **record_origin**: indicates that the row was generated through enrichment
- **license_source**: source licensing metadata

### Linguistic metadata
- **pos**: part of speech
- **tags**: lexical or source tags
- **idiom_confidence**: confidence or quality indicator from source preparation
- **is_idiom**: whether the expression is judged idiomatic
- **ambiguity_flag**: idiomatic ambiguity category
- **idiom_compositionality_level**: opacity / transparency level
- **idiom_register**: register such as formal, neutral, slang
- **idiom_domain**: semantic/domain category
- **learner_difficulty**: estimated learner difficulty

### Canonical meaning fields
- **idiom_canonical_meaning**: best English idiomatic meaning
- **idiom_canonical_meaning_arabic**: Arabic translation of canonical meaning

### Idiom-level semantic enhancement fields
- **hard_negative_idioms**: semantically confusable idioms useful for retrieval and ranking tasks
- **meaning_paraphrases_en**: English paraphrases of idiomatic meaning
- **meaning_paraphrases_ar**: Arabic paraphrases of idiomatic meaning
- **idiom_level_explanation_en**: English explanation of why the phrase is idiomatic
- **idiom_level_explanation_ar**: Arabic explanation of why the phrase is idiomatic

### Example-level fields
- **idiom_in_example**: generated English example sentence
- **idiom_in_example_arabic**: Arabic translation of the example sentence
- **idiom_in_example_meaning_en**: English meaning of the sentence in context
- **idiom_in_example_meaning_arabic**: Arabic translation of the sentence meaning
- **explanation_en**: English rationale for the example interpretation
- **explanation_ar**: Arabic rationale for the example interpretation

### Example-type / generation-control fields
- **is_example_idiom**: whether the example is idiomatic
- **example_usage_label**: idiomatic / literal / borderline label
- **context_type**: discourse style such as dialogue, narrative, formal, social_media, question, sarcastic, or adversarial
- **source_style**: generation style tag
- **row_type**: main example or adversarial example variant
- **is_adversarial_example**: whether the row is adversarial
- **adversarial_type**: adversarial subtype
- **expected_label**: expected interpretation label for adversarial rows
- **minimal_pair_id**: deterministic identifier linking paired idiomatic/literal contexts
- **paraphrase_group_id**: deterministic identifier linking related semantic paraphrase groups

### Technical / audit fields
- **example_language**: language of example text
- **meaning_language**: language of meaning text
- **is_generated_example**: whether the example was generated
- **enrichment_model**: model used for enrichment
- **enrichment_version**: enrichment version tag
- **validation_status**: validation / correction status

---

## Reproducibility Notes

This notebook is designed so that a user who downloads the IdiomX repository from GitHub can run the enrichment workflow with minimal changes.

### Assumptions
- the notebook is executed from inside the repository environment
- the `scripts/` directory exists
- the project folder contains a `data/` directory with the expected subfolders
- required Python dependencies are installed

### API Configuration
The preferred setup is to store API credentials in a local `.env` file and load them through the project configuration.

For users who do not use a local `.env` file, a commented notebook option is included below to set the API key manually inside the notebook.

### Execution Strategy
The notebook supports:
- **sample mode** for safe testing
- **full mode** for complete enrichment
- intermediate saving at each major pipeline stage
- final export into `data/final/`

---

### [1.1] Environment setup

This section defines the execution configuration and project paths used throughout the enrichment pipeline.

The notebook always uses the full pre-enrichment dataset generated in Notebook 1 as the single source of truth.

If `USE_SAMPLE_DATA = True`, a small in-memory subset is created from the full dataset for testing and debugging.  
The separate sample file saved in `data/final/` is kept only as a lightweight reviewer/demo artifact and is not used by the enrichment pipeline.

In [1]:
# [1.1] Environment setup and reproducibility

from pathlib import Path
import sys
import importlib

# ----------------------------
# Execution configuration
# ----------------------------
RUN_LLM_ENRICHMENT = False      # Set True only when running paid/API enrichment steps
USE_SAMPLE_DATA = True       # True = test on small in-memory subset from full dataset
OVERWRITE_BATCH = True
SAMPLE_SIZE = 2
SAMPLE_METHOD = "random"       # "head" or "random"

# ----------------------------
# Project paths
# ----------------------------
CURRENT_DIR = Path.cwd().resolve()
IDIOMX_ROOT = CURRENT_DIR.parent
DATA_DIR = IDIOMX_ROOT / "data"

FINAL_DIR = DATA_DIR / "final"
BATCHES_DIR = DATA_DIR / "batches"
RESULTS_DIR = DATA_DIR / "results"
ENRICHED_DIR = DATA_DIR / "enriched"

for p in [FINAL_DIR, BATCHES_DIR, RESULTS_DIR, ENRICHED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

if str(IDIOMX_ROOT) not in sys.path:
    sys.path.insert(0, str(IDIOMX_ROOT))

importlib.invalidate_caches()

# ----------------------------
# Input dataset from Notebook 1
# ----------------------------
FULL_PRE_FILE = FINAL_DIR / "idiomx_pre_enrichment.parquet"

# Reviewer/demo artifact only (not used as pipeline input)
REVIEW_SAMPLE_FILE = FINAL_DIR / "idiomx_pre_enrichment_sample.csv"

print("RUN_LLM_ENRICHMENT =", RUN_LLM_ENRICHMENT)
print("USE_SAMPLE_DATA    =", USE_SAMPLE_DATA)
print("OVERWRITE_BATCH    =", OVERWRITE_BATCH)
print("SAMPLE_SIZE        =", SAMPLE_SIZE)
print("SAMPLE_METHOD      =", SAMPLE_METHOD)
print("Project root       =", IDIOMX_ROOT)
print("Final dir          =", FINAL_DIR)
print("Full input file    =", FULL_PRE_FILE)
print("Scripts dir exists =", (IDIOMX_ROOT / "scripts").exists())
print("Full input exists  =", FULL_PRE_FILE.exists())
print("Reviewer sample    =", REVIEW_SAMPLE_FILE)
print("Reviewer sample exists =", REVIEW_SAMPLE_FILE.exists())

RUN_LLM_ENRICHMENT = True
USE_SAMPLE_DATA    = True
OVERWRITE_BATCH    = True
SAMPLE_SIZE        = 2
SAMPLE_METHOD      = random
Project root       = C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp
Final dir          = C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\final
Full input file    = C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\final\idiomx_pre_enrichment.parquet
Scripts dir exists = True
Full input exists  = True
Reviewer sample    = C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\final\idiomx_pre_enrichment_sample.csv
Reviewer sample exists = True


In [2]:
# [1.2] Optional manual API-key setup for notebook users
#
# Use this only if you do NOT already load the API key through your local .env file
# and project configuration.
#
# import os
# os.environ["OPENAI_API_KEY"] = "your_api_key_here"
#
# Note:
# In my local setup, the API key is already configured through the repository .env file,
# so this cell is normally not needed.

In [3]:
# Optional: install only if needed - uncomment the below lines
# import sys
# !{sys.executable} -m pip install python-dotenv

In [4]:
# [1.3] Import pipeline scripts

from scripts.en_01_prepare_enrichment_batch_requests_v2 import (
    create_sample_dataset,
    prepare_batch_requests,
)
from scripts.en_02_submit_batch_v2 import submit_batch
from scripts.en_03_check_existing_batch_v2 import check_batch
from scripts.en_04_download_existing_batch_results_v2 import download_results
from scripts.en_05_merge_batch_results_to_enriched_dataset_v2 import merge_results
from scripts.en_06_validate_dataset_v2 import validate_dataset
from scripts.en_07_verify_suspicious_rows_v2 import verify_suspicious_rows

### [1.4] Execution mode check

This step confirms whether the notebook is running in sample mode or full-dataset mode.

- **Sample mode** is recommended for testing and debugging
- **Full mode** should be used only when rebuilding the complete enrichment pipeline

In [5]:
# [1.4] Execution mode check

if USE_SAMPLE_DATA:
    print("Running SAMPLE dataset mode")
    print("This mode is recommended for testing, debugging, and quick validation.")
    print("To run the full enrichment pipeline, set USE_SAMPLE_DATA = False in cell [1.1].")
else:
    print("Running FULL dataset mode")
    print("This will:")
    print("- generate larger batch files")
    print("- consume API credits")
    print("- take significantly more time")


Running SAMPLE dataset mode
This mode is recommended for testing, debugging, and quick validation.
To run the full enrichment pipeline, set USE_SAMPLE_DATA = False in cell [1.1].


### [1.5] Enrichment pipeline file paths

This section defines the intermediate and output files used in the enrichment pipeline.

The pipeline always reads from the same full pre-enrichment dataset produced by Notebook 1.  
If sample mode is enabled, a small subset will be created dynamically from that full dataset during batch preparation.

This keeps the input contract simple while still supporting fast testing.

In [6]:
# [1.5] Enrichment pipeline file paths

# enrichment input
INPUT_PRE_FILE = FINAL_DIR / "idiomx_pre_enrichment.parquet"

if USE_SAMPLE_DATA:
    BATCH_JSONL_FILE = BATCHES_DIR / "idiomx_batch_sample_v2.jsonl"
    BATCH_INFO_FILE = BATCHES_DIR / "idiomx_batch_info_sample_v2.json"
    RESULTS_JSONL_FILE = RESULTS_DIR / "idiomx_results_sample_v2.jsonl"

    ENRICHED_CSV_FILE = ENRICHED_DIR / "idiomx_enriched_sample_v2.csv"
    ENRICHED_PARQUET_FILE = ENRICHED_DIR / "idiomx_enriched_sample_v2.parquet"
    
    VALIDATED_CSV_FILE = ENRICHED_DIR / "idiomx_enriched_validated_sample_v2.csv"
    ISSUES_CSV_FILE = ENRICHED_DIR / "idiomx_enriched_issues_sample_v2.csv"
    
    FINAL_VERIFIED_CSV_FILE = ENRICHED_DIR / "idiomx_enriched_final_sample_v2.csv"
    FINAL_VERIFIED_PARQUET_FILE = ENRICHED_DIR / "idiomx_enriched_final_sample_v2.parquet"

    FINAL_EXPORT_CSV_FILE = FINAL_DIR / "idiomx_final_sample_v2.csv"
    FINAL_EXPORT_PARQUET_FILE = FINAL_DIR / "idiomx_final_sample_v2.parquet"
else:
    BATCH_JSONL_FILE = BATCHES_DIR / "idiomx_batch_v2.jsonl"
    BATCH_INFO_FILE = BATCHES_DIR / "idiomx_batch_info_v2.json"
    RESULTS_JSONL_FILE = RESULTS_DIR / "idiomx_results_v2.jsonl"

    ENRICHED_CSV_FILE = ENRICHED_DIR / "idiomx_enriched_v2.csv"
    ENRICHED_PARQUET_FILE = ENRICHED_DIR / "idiomx_enriched_v2.parquet"
    
    VALIDATED_CSV_FILE = ENRICHED_DIR / "idiomx_enriched_validated_v2.csv"
    ISSUES_CSV_FILE = ENRICHED_DIR / "idiomx_enriched_issues_v2.csv"
    
    FINAL_VERIFIED_CSV_FILE = ENRICHED_DIR / "idiomx_enriched_final_v2.csv"
    FINAL_VERIFIED_PARQUET_FILE = ENRICHED_DIR / "idiomx_enriched_final_v2.parquet"
    
    FINAL_EXPORT_CSV_FILE = FINAL_DIR / "idiomx_final_v2.csv"
    FINAL_EXPORT_PARQUET_FILE = FINAL_DIR / "idiomx_final_v2.parquet"

print("Input pre-enrichment file:", INPUT_PRE_FILE)
print("Batch request file       :", BATCH_JSONL_FILE)
print("Batch info file          :", BATCH_INFO_FILE)
print("Results file             :", RESULTS_JSONL_FILE)
print("Intermediate enriched    :", ENRICHED_CSV_FILE)
print("Intermediate enriched Parquet :", ENRICHED_PARQUET_FILE)
print("Final verified CSV       :", FINAL_VERIFIED_CSV_FILE)
print("Final verified Parquet   :", FINAL_VERIFIED_PARQUET_FILE)
print("Final export CSV         :", FINAL_EXPORT_CSV_FILE)
print("Final export Parquet     :", FINAL_EXPORT_PARQUET_FILE)
print("\n--- Sanity Check ---")
print("Input exists      :", INPUT_PRE_FILE.exists())
print("Batch JSON exists :", BATCH_JSONL_FILE.exists())
print("Results exist     :", RESULTS_JSONL_FILE.exists())

Input pre-enrichment file: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\final\idiomx_pre_enrichment.parquet
Batch request file       : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\batches\idiomx_batch_sample_v2.jsonl
Batch info file          : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\batches\idiomx_batch_info_sample_v2.json
Results file             : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\results\idiomx_results_sample_v2.jsonl
Intermediate enriched    : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\enriched\idiomx_enriched_sample_v2.csv
Intermediate enriched Parquet : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\enriched\idiomx_enriched_sample_v2.parquet
Final verified CSV       : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\enriched\idiomx_enriched_final_sample_v2.csv
Final verified Parquet   : C:\Users\

### Input Dataset

This notebook expects the pre-enrichment dataset generated from the dataset construction pipeline:

data/final/idiomx_pre_enrichment.csv

If the file is missing, please run Notebook 1 first.

In [7]:
# [1.6] check Dataset exist

assert INPUT_PRE_FILE.exists(), "Pre-enrichment dataset not found. Run Notebook 1 first."
print("Using input file:", INPUT_PRE_FILE)

Using input file: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\final\idiomx_pre_enrichment.parquet


### [2.1] Prepare batch requests

This step converts the pre-enrichment dataset into a JSONL file suitable for batch submission to the LLM API.

The step is safe to rerun because:
- it does not call the API
- it only generates the request file
- overwrite behavior is controlled explicitly

A rough token estimate is also reported for planning and cost awareness.

In [8]:
# [2.1] Prepare batch requests

import pandas as pd
from pathlib import Path

ESTIMATED_TOKENS_PER_REQUEST = 300

assert INPUT_PRE_FILE.exists(), f"Missing input dataset: {INPUT_PRE_FILE}"

# Load full source once
input_df = pd.read_parquet(INPUT_PRE_FILE)

# Apply in-memory sample for planning/statistics
if USE_SAMPLE_DATA:
    if SAMPLE_METHOD == "head":
        effective_df = input_df.head(SAMPLE_SIZE).copy()
    else:
        effective_df = input_df.sample(min(SAMPLE_SIZE, len(input_df)), random_state=42).copy()
else:
    effective_df = input_df

total_requests = len(effective_df)

print("Input dataset (source):", INPUT_PRE_FILE)
print("Execution mode         :", "sample" if USE_SAMPLE_DATA else "full")
print("Effective requests     :", total_requests)

# Handle overwrite behavior
if BATCH_JSONL_FILE.exists():
    if OVERWRITE_BATCH:
        print(f"Overwriting existing batch file: {BATCH_JSONL_FILE}")
    else:
        print(f"Batch file already exists: {BATCH_JSONL_FILE}")
        print("Skipping regeneration because OVERWRITE_BATCH = False")
        batch_file = BATCH_JSONL_FILE

if not BATCH_JSONL_FILE.exists() or OVERWRITE_BATCH:
    print("Preparing batch JSONL file...")
    batch_file = prepare_batch_requests(
        input_file=INPUT_PRE_FILE,
        output_jsonl=BATCH_JSONL_FILE,
        use_sample=USE_SAMPLE_DATA,
        sample_size=SAMPLE_SIZE,
        sample_method=SAMPLE_METHOD,
    )

print("Batch file:", batch_file)
print("Batch file exists:", Path(batch_file).exists())

estimated_total_tokens = total_requests * ESTIMATED_TOKENS_PER_REQUEST

print("\nEstimated usage:")
print("Estimated tokens per request:", ESTIMATED_TOKENS_PER_REQUEST)
print("Estimated total tokens:", estimated_total_tokens)
print("Estimated total tokens (millions):", round(estimated_total_tokens / 1_000_000, 4))

Input dataset (source): C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\final\idiomx_pre_enrichment.parquet
Execution mode         : sample
Effective requests     : 2
Overwriting existing batch file: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\batches\idiomx_batch_sample_v2.jsonl
Preparing batch JSONL file...
[WARNING] Output batch file already exists and will be overwritten: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\batches\idiomx_batch_sample_v2.jsonl
Saved batch file to: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\batches\idiomx_batch_sample_v2.jsonl
Input dataset: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\final\idiomx_pre_enrichment.parquet
Total requests: 2
Batch file: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\batches\idiomx_batch_sample_v2.jsonl
Batch file exists: True

Estimated usage:
Estimated tokens per request: 300
Esti

### Batch Preparation Completed

The JSONL batch file has been successfully created.

- Each line represents one API request
- The file is ready for submission to the batch API
- Total number of requests equals the number of input rows

Next step: Submit the batch job to the LLM.

---

---
### [3.1] Submit batch job to the LLM API

This step uploads the prepared JSONL batch file to the LLM batch API.

The API will:
- validate the request file
- queue the batch for asynchronous processing
- return a unique batch identifier

This step **will consume API credits**, so it should only be run when the batch file has been reviewed and the enrichment configuration is final.

In [9]:
# [3.1] Submit batch job to the LLM API

from pathlib import Path

assert BATCH_JSONL_FILE.exists(), f"Batch JSONL file not found: {BATCH_JSONL_FILE}"

batch_path = Path(BATCH_JSONL_FILE)
batch_size_bytes = batch_path.stat().st_size
batch_size_mb = round(batch_size_bytes / (1024 * 1024), 4)

with open(batch_path, "r", encoding="utf-8") as f:
    batch_request_count = sum(1 for _ in f)

print("Batch input file      :", BATCH_JSONL_FILE)
print("Batch info file       :", BATCH_INFO_FILE)
print("RUN_LLM_ENRICHMENT    :", RUN_LLM_ENRICHMENT)
print("USE_SAMPLE_DATA       :", USE_SAMPLE_DATA)
print("OVERWRITE_BATCH       :", OVERWRITE_BATCH)
print("Batch file exists     :", batch_path.exists())
print("Batch request count   :", batch_request_count)
print("Batch file size (MB)  :", batch_size_mb)

# Safety checks
if USE_SAMPLE_DATA and batch_request_count > SAMPLE_SIZE:
    raise ValueError(
        f"Sample mode is enabled, but batch file contains {batch_request_count} requests "
        f"instead of expected <= {SAMPLE_SIZE}. Regenerate the batch file first."
    )

if not USE_SAMPLE_DATA and batch_request_count < 100:
    print("Warning: full mode is enabled, but batch request count looks unusually small.")

if RUN_LLM_ENRICHMENT:
    print("\nSubmitting batch job to the LLM API...")

    batch_id = submit_batch(
        batch_file=BATCH_JSONL_FILE,
        batch_info_file=BATCH_INFO_FILE,
        use_sample=USE_SAMPLE_DATA,
        stage_name="idiomx_pre_enrichment_to_enriched_v2"
    )

    print("\nBatch submitted successfully.")
    print("Batch ID              :", batch_id)
    print("Batch info saved to   :", BATCH_INFO_FILE)
    print("Batch info file exists:", BATCH_INFO_FILE.exists())
else:
    print("\nBatch submission skipped because RUN_LLM_ENRICHMENT = False.")
    print("Set RUN_LLM_ENRICHMENT = True in cell [1.1] only when you are ready to consume API credits.")

Batch input file      : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\batches\idiomx_batch_sample_v2.jsonl
Batch info file       : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\batches\idiomx_batch_info_sample_v2.json
RUN_LLM_ENRICHMENT    : True
USE_SAMPLE_DATA       : True
OVERWRITE_BATCH       : True
Batch file exists     : True
Batch request count   : 2
Batch file size (MB)  : 0.0151

Submitting batch job to the LLM API...
Uploaded file ID: file-SzFvyRCvnJHgAWQtSZSiTA
Batch ID: batch_69d0ae96e6f08190b6e5d6ef66c83f0e
Batch status: validating
Batch info saved to: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\batches\idiomx_batch_info_sample_v2.json

Batch submitted successfully.
Batch ID              : batch_69d0ae96e6f08190b6e5d6ef66c83f0e
Batch info saved to   : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\batches\idiomx_batch_info_sample_v2.json
Batch info file exists: True


### Batch Submitted Successfully

The batch job has been submitted.

- A unique batch ID has been generated
- The batch is now being processed asynchronously by the API

Next step: Monitor the batch status until completion.

---

---
### [3.2] Monitor batch status

This step continuously checks the status of the submitted batch job.

Possible statuses include:
- validating
- in_progress
- completed
- failed
- cancelled
- expired

The monitoring loop will poll the batch status every few seconds until:
- the batch is completed successfully, or
- a terminal state is reached (failed / cancelled / expired), or
- a maximum wait time is exceeded.

This step is safe to run multiple times.

In [10]:
# [3.2] Monitor batch status safely

import time

assert BATCH_INFO_FILE.exists(), f"Batch info file not found: {BATCH_INFO_FILE}"

POLL_INTERVAL = 30          # seconds
MAX_WAIT_MINUTES = 60       # timeout prevents infinite loop

max_attempts = (MAX_WAIT_MINUTES * 60) // POLL_INTERVAL
attempt = 0

print("Monitoring batch status...")
print(f"Polling every {POLL_INTERVAL} seconds (max {MAX_WAIT_MINUTES} minutes)\n")

while attempt < max_attempts:
    batch = check_batch(
        batch_info_file=BATCH_INFO_FILE,
        use_sample=USE_SAMPLE_DATA,
    )

    if batch is None:
        print("Failed to retrieve batch status.")
        break

    status = getattr(batch, "status", None)
    print(f"[Attempt {attempt+1}] Status: {status}")

    if status == "completed":
        print("\nBatch completed successfully.")
        break

    elif status in ["failed", "cancelled", "expired"]:
        print(f"\nBatch stopped with status: {status}")
        break

    else:
        attempt += 1
        print(f"Waiting {POLL_INTERVAL} seconds...\n")
        time.sleep(POLL_INTERVAL)

else:
    print("\nTimeout reached before batch completion.")

Monitoring batch status...
Polling every 30 seconds (max 60 minutes)

Batch still in progress.
Batch ID: batch_69d0ae96e6f08190b6e5d6ef66c83f0e
Status: validating
Created at: 1775283862
In progress at: None
Completed at: None
Output file ID: None
Error file ID: None
[Attempt 1] Status: validating
Waiting 30 seconds...

Batch still in progress.
Batch ID: batch_69d0ae96e6f08190b6e5d6ef66c83f0e
Status: in_progress
Created at: 1775283862
In progress at: 1775283864
Completed at: None
Output file ID: None
Error file ID: None
[Attempt 2] Status: in_progress
Waiting 30 seconds...

Batch still in progress.
Batch ID: batch_69d0ae96e6f08190b6e5d6ef66c83f0e
Status: in_progress
Created at: 1775283862
In progress at: 1775283864
Completed at: None
Output file ID: None
Error file ID: None
[Attempt 3] Status: in_progress
Waiting 30 seconds...

Batch still in progress.
Batch ID: batch_69d0ae96e6f08190b6e5d6ef66c83f0e
Status: in_progress
Created at: 1775283862
In progress at: 1775283864
Completed at: Non

KeyboardInterrupt: 

### Batch Status Retrieved

The current batch status has been retrieved.

- If status is `completed`, you can proceed to download results
- Otherwise, wait and check again later

Next step: Download results after completion.

---

---
### [4.1] Download batch results

Once the batch job is completed, the generated outputs are downloaded from the batch API.

The downloaded results are stored as a JSONL file, where each line corresponds to one model response associated with one input request.

These results will be used in the next step to reconstruct the enriched dataset.

In [11]:
# [4.1] Download batch results

from pathlib import Path

assert BATCH_INFO_FILE.exists(), f"Batch info file not found: {BATCH_INFO_FILE}"

print("Batch info file :", BATCH_INFO_FILE)
print("Results output  :", RESULTS_JSONL_FILE)

results_path = download_results(
    batch_info_file=BATCH_INFO_FILE,
    output_path=RESULTS_JSONL_FILE,
    use_sample=USE_SAMPLE_DATA,
)

results_path = Path(results_path)

print("\nDownload completed.")
print("Results path      :", results_path)
print("Results file exists:", results_path.exists())
print("Results file size  :", results_path.stat().st_size if results_path.exists() else "N/A")

Batch info file : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\batches\idiomx_batch_info_sample_v2.json
Results output  : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\results\idiomx_results_sample_v2.jsonl
Downloaded results to: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\results\idiomx_results_sample_v2.jsonl

Download completed.
Results path      : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\results\idiomx_results_sample_v2.jsonl
Results file exists: True
Results file size  : 58945


### Results Downloaded

The batch results have been successfully downloaded.

- Each line corresponds to one processed request
- The data is now ready for merging with the original dataset

Next step: Merge results into the dataset.

---

---
### [5.1] Merge results with original dataset

In this step, we combine:
- the original pre-enrichment dataset
- the generated LLM outputs (JSONL)

This produces a fully enriched dataset.

The final dataset includes:
- original fields
- generated examples
- explanations and paraphrases
- metadata and validation fields

The output is saved in both CSV and Parquet formats for flexibility and performance.
---

In [11]:
# [5.1] Merge batch results into enriched dataset

from pathlib import Path
import pandas as pd

print("Merging batch results into enriched dataset...")

# ----------------------------
# Validate required inputs
# ----------------------------
assert INPUT_PRE_FILE.exists(), f"Missing input dataset: {INPUT_PRE_FILE}"
assert RESULTS_JSONL_FILE.exists(), f"Missing results file: {RESULTS_JSONL_FILE}"

print("Input dataset :", INPUT_PRE_FILE)
print("Results file  :", RESULTS_JSONL_FILE)

# ----------------------------
# Run merge
# ----------------------------
merge_results(
    raw_file=INPUT_PRE_FILE,              # parquet input
    results_jsonl=RESULTS_JSONL_FILE,
    output_csv=ENRICHED_CSV_FILE,
    output_parquet=ENRICHED_PARQUET_FILE,
    use_sample=USE_SAMPLE_DATA,
)

print("\nMerge completed successfully.")
print("CSV output     :", ENRICHED_CSV_FILE)
print("Parquet output :", ENRICHED_PARQUET_FILE)

# ----------------------------
# Quick validation (lightweight)
# ----------------------------
if ENRICHED_PARQUET_FILE.exists():
    merged_df = pd.read_parquet(ENRICHED_PARQUET_FILE)
else:
    merged_df = pd.read_csv(ENRICHED_CSV_FILE, low_memory=False)

print("\nMerged dataset shape:", merged_df.shape)
display(merged_df.head(3))

Merging batch results into enriched dataset...
Input dataset : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\final\idiomx_pre_enrichment.parquet
Results file  : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\results\idiomx_results_sample_v2.jsonl
Raw file: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\final\idiomx_pre_enrichment.parquet
Results file: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\results\idiomx_results_sample_v2.jsonl
Saved CSV: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\enriched\idiomx_enriched_sample_v2.csv
Saved Parquet: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\enriched\idiomx_enriched_sample_v2.parquet
Total rows: 28

example_usage_label distribution:
example_usage_label
literal       13
idiomatic     12
borderline     3
Name: count, dtype: int64

row_type distribution:
row_type
main_example             24
adversarial

,idiom_id,idiom_canonical,idiom_surface,example,idiom_canonical_meaning,source,source_type,pos,tags,idiom_confidence,...,idiom_level_explanation_en,idiom_level_explanation_ar,explanation_en,explanation_ar,minimal_pair_id,paraphrase_group_id,is_adversarial_example,adversarial_type,expected_label,row_type
0,idiomx_e3e7571a5cef,turkey shoot,turkey shoot,"In the Battle of the Marianas, which pilots ca...",A situation where one side faces overwhelming ...,kaikki_wiktionary,dictionary,noun,"broadly, idiomatic",high,...,The phrase 'turkey shoot' is idiomatic because...,"تعبير ""turkey shoot"" تعبير اصطلاحي لأنه يصف بش...",This example clearly uses the phrase figurativ...,يستخدم هذا المثال العبارة بشكل مجازي لتوصيل مع...,pair_9d8101efff7b,paraphrase_305734ad2f7a,0,,idiomatic,main_example
1,idiomx_e3e7571a5cef,turkey shoot,turkey shoot,"In the Battle of the Marianas, which pilots ca...",A situation where one side faces overwhelming ...,kaikki_wiktionary,dictionary,noun,"broadly, idiomatic",high,...,The phrase 'turkey shoot' is idiomatic because...,"تعبير ""turkey shoot"" تعبير اصطلاحي لأنه يصف بش...","Used literally, the phrase refers to an actual...",باستخدام حرفي، تشير العبارة إلى حدث صيد ديك رو...,pair_9d8101efff7b,paraphrase_305734ad2f7a,0,,literal,main_example
2,idiomx_e3e7571a5cef,turkey shoot,turkey shoot,"In the Battle of the Marianas, which pilots ca...",A situation where one side faces overwhelming ...,kaikki_wiktionary,dictionary,noun,"broadly, idiomatic",high,...,The phrase 'turkey shoot' is idiomatic because...,"تعبير ""turkey shoot"" تعبير اصطلاحي لأنه يصف بش...",This narrative uses the phrase figuratively to...,يستخدم هذا السرد العبارة مجازيًا للدلالة على ه...,pair_dd862b5ab208,paraphrase_305734ad2f7a,0,,idiomatic,main_example


### Merge Completed

The enriched dataset has been created successfully.

- All generated outputs are now aligned with the original rows
- The dataset is ready for validation

Next step: Validate the dataset.


---
### [6.1] QA diagnostics after merge

This step evaluates the quality and completeness of the enriched dataset.

We verify:
- dataset size and structure
- distribution of row types
- label balance (idiomatic vs literal)
- completeness of generated outputs
- per-idiom coverage (main and adversarial examples)

These diagnostics help identify missing or low-quality enrichments before validation and final export.

In [12]:
# [6.1] QA diagnostics after merge

import pandas as pd

# ----------------------------
# Load dataset (parquet-first)
# ----------------------------
if ENRICHED_PARQUET_FILE.exists():
    merged_df = pd.read_parquet(ENRICHED_PARQUET_FILE)
else:
    merged_df = pd.read_csv(ENRICHED_CSV_FILE, low_memory=False)

print("Merged shape:", merged_df.shape)

# ----------------------------
# Row type distribution
# ----------------------------
print("\nrow_type counts:")
print(merged_df["row_type"].value_counts(dropna=False))

# ----------------------------
# Label distribution
# ----------------------------
print("\nexample_usage_label counts:")
print(merged_df["example_usage_label"].value_counts(dropna=False))

# ----------------------------
# Missing values check
# ----------------------------
print("\nMissing values (top columns):")
missing = merged_df.isna().sum().sort_values(ascending=False)
print(missing.head(10))

# ----------------------------
# Coverage per idiom
# ----------------------------
grouped = merged_df.groupby("idiom_id").agg(
    total_rows=("idiom_id", "size"),
    main_rows=("row_type", lambda s: (s == "main_example").sum()),
    adv_rows=("row_type", lambda s: s.astype(str).str.startswith("adversarial_example").sum()),
)

# ----------------------------
# Coverage diagnostics
# ----------------------------
incomplete_main = (grouped["main_rows"] < 12).sum()
incomplete_adv = (grouped["adv_rows"] < 2).sum()

print("\nCoverage diagnostics:")
print("Idioms with incomplete main rows (<12):", incomplete_main)
print("Idioms with incomplete adversarial rows (<2):", incomplete_adv)

# ----------------------------
# LLM failure detection (important)
# ----------------------------
if "generated_sentence" in merged_df.columns:
    empty_gen = merged_df["generated_sentence"].isna().sum()
    print("\nEmpty generated_sentence rows:", empty_gen)

if "explanation" in merged_df.columns:
    empty_exp = merged_df["explanation"].isna().sum()
    print("Empty explanation rows:", empty_exp)

# ----------------------------
# Coverage percentage (for paper)
# ----------------------------
total_idioms = grouped.shape[0]

print("\nCoverage %:")
print("Main coverage   :", round(100 * (1 - incomplete_main / total_idioms), 2), "%")
print("Adversarial coverage:", round(100 * (1 - incomplete_adv / total_idioms), 2), "%")

# ----------------------------
# Show weakest cases
# ----------------------------
print("\nLowest coverage idioms:")
display(grouped.sort_values(["main_rows", "adv_rows"]).head(20))

Merged shape: (28, 47)

row_type counts:
row_type
main_example             24
adversarial_example_1     2
adversarial_example_2     2
Name: count, dtype: int64

example_usage_label counts:
example_usage_label
literal       13
idiomatic     12
borderline     3
Name: count, dtype: int64

Missing values (top columns):
source_url                28
idiom_surface              4
minimal_pair_id            4
is_example_idiom           3
meaning_paraphrases_en     0
example_usage_label        0
is_generated_example       0
enrichment_model           0
enrichment_version         0
validation_status          0
dtype: int64

Coverage diagnostics:
Idioms with incomplete main rows (<12): 0
Idioms with incomplete adversarial rows (<2): 0

Coverage %:
Main coverage   : 100.0 %
Adversarial coverage: 100.0 %

Lowest coverage idioms:


,total_rows,main_rows,adv_rows
idiom_id,,,
idiomx_de44334e4d0c,14,12,2
idiomx_e3e7571a5cef,14,12,2


---
## Step 5. Enriched Dataset Insights (v2)

### Dataset Overview

The LLM-enriched IdiomX dataset (v2) was successfully generated and merged, resulting in:

- **Total rows:** 179,883  
- **Total idioms:** 12,853  
- **Columns:** 47  

Each idiom is expanded into multiple structured examples, including both standard and adversarial cases.

---

### Row Type Distribution

| Row Type                | Count   |
|------------------------|--------:|
| Main Examples          | 154,189 |
| Adversarial Example 1  | 12,847  |
| Adversarial Example 2  | 12,847  |

The dataset includes a consistent adversarial component, enhancing robustness for downstream tasks such as classification and retrieval.

---

### Example Usage Distribution

| Label       | Count   |
|------------|--------:|
| Literal     | 85,627  |
| Idiomatic   | 80,693  |
| Borderline  | 13,557  |

The dataset is well-balanced between literal and idiomatic usage.  
The inclusion of **borderline cases** introduces realistic ambiguity, improving generalization.

---

### Structural Completeness Check

Out of 12,853 idioms:

- **Idioms with incomplete main examples (<12):** 6  
- **Idioms with incomplete adversarial examples (<2):** 6  

This represents:

- **~0.046% structural inconsistency**

The vast majority of generated outputs strictly follow the expected schema.  
Minor inconsistencies are traceable and isolated to a very small subset of idioms.

---

###  Data Quality Observations

- JSON repair successfully recovered partially malformed responses.
- A small number of cases returned fewer examples than expected.
- No batch-level failures occurred (**0 failed requests**).
- All idioms were processed successfully.

---

### Key Takeaways

- The enrichment pipeline achieved **near-complete structured generation at scale**.
- The dataset provides:
  - High linguistic diversity
  - Balanced semantic labels
  - Adversarial robustness
  - Minimal structural noise

---

### Conclusion

The IdiomX v2 dataset is **production-ready and suitable for deep learning tasks**, particularly:

- Task 1: Idiom Detection  
- Task 2: Context → Idiom Retrieval  
- Task 3: Cross-lingual Mapping  

The dataset also supports advanced evaluation scenarios due to the presence of adversarial and borderline examples.

---

---

---
### [6.1] Validate enriched dataset

This step performs structured validation on the enriched dataset.

It verifies:
- required fields are present
- label values are valid and consistent
- generated outputs follow expected formats
- missing or malformed rows are identified

Invalid or suspicious rows are flagged and saved separately for review.

The validated dataset is then used for final export.

In [13]:
# [6.2] Validate enriched dataset

from pathlib import Path
import pandas as pd

print("Validating enriched dataset...")

# ----------------------------
# Validation input must be CSV
# ----------------------------
assert ENRICHED_CSV_FILE.exists(), f"Missing enriched CSV file: {ENRICHED_CSV_FILE}"

input_file = ENRICHED_CSV_FILE
merged_df = pd.read_csv(input_file, low_memory=False)

print("Input file used:", input_file)
print("Dataset shape  :", merged_df.shape)

# ----------------------------
# Run validation
# ----------------------------
validated_df, issues_df = validate_dataset(
    input_file=input_file,
    output_validated_csv=VALIDATED_CSV_FILE,
    output_issues_csv=ISSUES_CSV_FILE,
    use_sample=USE_SAMPLE_DATA,
)

print("\nValidation completed successfully.")
print("Validated dataset shape:", validated_df.shape)
print("Issues dataset shape   :", issues_df.shape)

print("Validated CSV saved to :", VALIDATED_CSV_FILE)
print("Issues CSV saved to    :", ISSUES_CSV_FILE)

print("\nValidation status counts:")
print(validated_df["validation_status"].value_counts(dropna=False))

if not issues_df.empty:
    print("\nTop issue types:")
    print(issues_df["problem"].value_counts().head(10))

critical_cols = ["idiom_id", "generated_sentence", "explanation"]
existing_cols = [c for c in critical_cols if c in validated_df.columns]

if existing_cols:
    print("\nMissing critical fields:")
    print(validated_df[existing_cols].isna().sum())

display(validated_df.head(3))
display(issues_df.head(10))

Validating enriched dataset...
Input file used: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\enriched\idiomx_enriched_sample_v2.csv
Dataset shape  : (28, 47)
Validated dataset saved to: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\enriched\idiomx_enriched_validated_sample_v2.csv
Issues saved to: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\enriched\idiomx_enriched_issues_sample_v2.csv
validation_status
valid    28
Name: count, dtype: int64

Validation success rate: 100.00%

Validation completed successfully.
Validated dataset shape: (28, 47)
Issues dataset shape   : (0, 0)
Validated CSV saved to : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\enriched\idiomx_enriched_validated_sample_v2.csv
Issues CSV saved to    : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\enriched\idiomx_enriched_issues_sample_v2.csv

Validation status counts:
validation_status
valid    28
Nam

,idiom_id,idiom_canonical,idiom_surface,example,idiom_canonical_meaning,source,source_type,pos,tags,idiom_confidence,...,idiom_level_explanation_en,idiom_level_explanation_ar,explanation_en,explanation_ar,minimal_pair_id,paraphrase_group_id,is_adversarial_example,adversarial_type,expected_label,row_type
0,idiomx_e3e7571a5cef,turkey shoot,turkey shoot,"In the Battle of the Marianas, which pilots ca...",A situation where one side faces overwhelming ...,kaikki_wiktionary,dictionary,noun,"broadly, idiomatic",high,...,The phrase 'turkey shoot' is idiomatic because...,"تعبير ""turkey shoot"" تعبير اصطلاحي لأنه يصف بش...",This example clearly uses the phrase figurativ...,يستخدم هذا المثال العبارة بشكل مجازي لتوصيل مع...,pair_9d8101efff7b,paraphrase_305734ad2f7a,0,NaN,idiomatic,main_example
1,idiomx_e3e7571a5cef,turkey shoot,turkey shoot,"In the Battle of the Marianas, which pilots ca...",A situation where one side faces overwhelming ...,kaikki_wiktionary,dictionary,noun,"broadly, idiomatic",high,...,The phrase 'turkey shoot' is idiomatic because...,"تعبير ""turkey shoot"" تعبير اصطلاحي لأنه يصف بش...","Used literally, the phrase refers to an actual...",باستخدام حرفي، تشير العبارة إلى حدث صيد ديك رو...,pair_9d8101efff7b,paraphrase_305734ad2f7a,0,NaN,literal,main_example
2,idiomx_e3e7571a5cef,turkey shoot,turkey shoot,"In the Battle of the Marianas, which pilots ca...",A situation where one side faces overwhelming ...,kaikki_wiktionary,dictionary,noun,"broadly, idiomatic",high,...,The phrase 'turkey shoot' is idiomatic because...,"تعبير ""turkey shoot"" تعبير اصطلاحي لأنه يصف بش...",This narrative uses the phrase figuratively to...,يستخدم هذا السرد العبارة مجازيًا للدلالة على ه...,pair_dd862b5ab208,paraphrase_305734ad2f7a,0,NaN,idiomatic,main_example


""


### Validation Completed

The dataset has been validated.

- Clean rows are marked as valid
- Problematic rows are separated for inspection

Next step: Verify and correct suspicious rows.

---
## 6. Dataset Validation Insights

###  Validation Overview

The enriched IdiomX v2 dataset was validated to assess structural consistency, semantic correctness, and labeling quality.

- **Total rows:** 179,883  
- **Valid rows:** 173,314  
- **Rows requiring review:** 6,569  

**Validation success rate: 96.35%**

---

### Validation Status Distribution

| Status        | Count   |
|--------------|--------:|
| Valid         | 173,314 |
| Needs Review  | 6,569   |

✔ The majority of the dataset meets all validation criteria.  
✔ A small portion (~3.65%) is flagged for review.

---

### Top Detected Issues

| Issue Type                                      | Count |
|------------------------------------------------|------:|
| surface_not_in_example                          | 6,084 |
| unexpected_context_pair_count_per_idiom         | 418   |
| missing_idiom_canonical                         | 350   |
| unexpected_idiomatic_main_count_per_idiom       | 317   |
| unexpected_literal_main_count_per_idiom         | 317   |
| label_boolean_mismatch_idiomatic                | 78    |
| example_too_short                              | 38    |
| label_boolean_mismatch_literal                 | 13    |
| invalid_main_example_usage_label               | 6     |
| unexpected_main_example_count_per_idiom        | 6     |

---

### Interpretation of Issues

#### 1. Surface Not in Example (Majority)
- The most frequent issue (≈93% of flagged rows).
- Indicates that the **idiom surface form does not explicitly appear in the generated sentence**.
- This often occurs due to:
  - paraphrasing
  - implicit idiomatic usage
  - generation drift

This is a **soft error**, not a critical failure.

---

#### 2. Structural Count Mismatches
- Slight deviations from expected:
  - 12 main examples
  - 2 adversarial examples
- Already observed during merge stage.

Impact is minimal and isolated.

---

#### 3. Missing or Incomplete Fields
- Missing canonical forms or short examples.
- Very low frequency.

These are **minor and recoverable issues**.

---

### Quality Assessment

- **High structural integrity:** >96% valid rows  
- **Low corruption rate:** <4% flagged  
- **Zero batch failures:** all requests processed successfully  
- **Issues are mostly non-critical**

---

### Key Insight

The validation results confirm that:

> The LLM enrichment pipeline produces **high-quality, structured, and scalable data** with minimal post-processing requirements.

---

### Recommendation

- Proceed with the dataset as the **primary training source (v2)**  
- Optionally:
  - filter or correct `needs_review` rows
  - use them for **error analysis in the research paper**

---

### Research Value

This validation step demonstrates:

- robustness of large-scale LLM data generation  
- effectiveness of post-generation validation  
- practical feasibility of semi-automated dataset construction  

---

---

---
### [7.1] Verify and correct suspicious rows

This step reviews and corrects rows flagged during validation.

It performs:
- targeted re-evaluation of suspicious samples
- correction of malformed or low-quality outputs
- tracking of all changes via an audit log

Outputs include:
- final corrected dataset
- rows requiring manual review
- corrected rows
- verification log (before/after changes)

This ensures the final dataset reaches high quality and consistency standards.

In [19]:
# [7.1] Verify and correct suspicious rows

from pathlib import Path
import pandas as pd
import importlib
import scripts.en_07_verify_suspicious_rows_v2

importlib.reload(scripts.en_07_verify_suspicious_rows_v2)
from scripts.en_07_verify_suspicious_rows_v2 import verify_suspicious_rows
print("Verifying suspicious rows...")

# ----------------------------
# Load validated dataset (parquet-first)
# ----------------------------
if FINAL_VERIFIED_PARQUET_FILE.exists():
    input_file = FINAL_VERIFIED_PARQUET_FILE
elif VALIDATED_CSV_FILE.exists():
    input_file = VALIDATED_CSV_FILE
else:
    raise FileNotFoundError("No validated dataset found.")

print("Input file used:", input_file)

# ----------------------------
# Define audit files
# ----------------------------
if USE_SAMPLE_DATA:
    REVIEW_ROWS_FILE = ENRICHED_DIR / "idiomx_review_rows_sample_v2.csv"
    CORRECTED_ROWS_FILE = ENRICHED_DIR / "idiomx_corrected_rows_sample_v2.csv"
    VERIFICATION_LOG_FILE = ENRICHED_DIR / "idiomx_verification_log_sample_v2.csv"
else:
    REVIEW_ROWS_FILE = ENRICHED_DIR / "idiomx_review_rows_v2.csv"
    CORRECTED_ROWS_FILE = ENRICHED_DIR / "idiomx_corrected_rows_v2.csv"
    VERIFICATION_LOG_FILE = ENRICHED_DIR / "idiomx_verification_log_v2.csv"

# Ensure directories exist
for p in [
    REVIEW_ROWS_FILE.parent,
    CORRECTED_ROWS_FILE.parent,
    VERIFICATION_LOG_FILE.parent,
]:
    p.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Run verification
# ----------------------------
final_df, review_rows_df, corrected_rows_df, verification_log_df = verify_suspicious_rows(
    input_csv=input_file,
    output_csv=FINAL_VERIFIED_CSV_FILE,
    review_rows_csv=REVIEW_ROWS_FILE,
    corrected_rows_csv=CORRECTED_ROWS_FILE,
    verification_log_csv=VERIFICATION_LOG_FILE,
    use_sample=USE_SAMPLE_DATA,
    model_name="gpt-4.1-mini",
)

# ----------------------------
# Save parquet version (important)
# ----------------------------
final_df.to_parquet(FINAL_VERIFIED_PARQUET_FILE, index=False)

print("\nVerification completed successfully.")
print("Final dataset shape      :", final_df.shape)
print("Review rows shape        :", review_rows_df.shape)
print("Corrected rows shape     :", corrected_rows_df.shape)
print("Verification log shape   :", verification_log_df.shape)

print("\nOutputs:")
print("Final CSV                :", FINAL_VERIFIED_CSV_FILE)
print("Final Parquet            :", FINAL_VERIFIED_PARQUET_FILE)
print("Review rows              :", REVIEW_ROWS_FILE)
print("Corrected rows           :", CORRECTED_ROWS_FILE)
print("Verification log         :", VERIFICATION_LOG_FILE)

# ----------------------------
# Summary
# ----------------------------
print("\nFinal validation status:")
print(final_df["validation_status"].value_counts(dropna=False))

# ----------------------------
# Correction stats (important)
# ----------------------------
print("\nCorrection stats:")
print("Rows sent for review :", len(review_rows_df))
print("Rows corrected       :", len(corrected_rows_df))

# ----------------------------
# Preview
# ----------------------------
print("\nPreview: final dataset")
display(final_df.head(3))

if not review_rows_df.empty:
    print("\nPreview: rows sent for review")
    display(review_rows_df.head(10))
else:
    print("\nPreview: rows sent for review -> none")

if not corrected_rows_df.empty:
    print("\nPreview: corrected rows")
    display(corrected_rows_df.head(10))
else:
    print("\nPreview: corrected rows -> none")

if not verification_log_df.empty:
    print("\nPreview: verification log")
    display(verification_log_df.head(10))
else:
    print("\nPreview: verification log -> none")
    
print("\nVerification result summary:")
print("- Final dataset rows      :", len(final_df))
print("- Rows sent for review    :", len(review_rows_df))
print("- Rows corrected          :", len(corrected_rows_df))
print("- Verification log rows   :", len(verification_log_df))

Verifying suspicious rows...
Input file used: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\enriched\idiomx_enriched_final_sample_v2.parquet
[INFO] Resuming from checkpoint CSV: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\sample\idiomx_verification_checkpoint_sample_v2.csv
Rows marked for review: 0
No suspicious rows found. Saved unchanged dataset to: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\enriched\idiomx_enriched_final_sample_v2.csv

Verification completed successfully.
Final dataset shape      : (28, 47)
Review rows shape        : (0, 47)
Corrected rows shape     : (0, 47)
Verification log shape   : (0, 7)

Outputs:
Final CSV                : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\enriched\idiomx_enriched_final_sample_v2.csv
Final Parquet            : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\enriched\idiomx_enriched_final_sample_v2.parquet
Revie

,idiom_id,idiom_canonical,idiom_surface,example,idiom_canonical_meaning,source,source_type,pos,tags,idiom_confidence,...,idiom_level_explanation_en,idiom_level_explanation_ar,explanation_en,explanation_ar,minimal_pair_id,paraphrase_group_id,is_adversarial_example,adversarial_type,expected_label,row_type
0,idiomx_e3e7571a5cef,turkey shoot,turkey shoot,"In the Battle of the Marianas, which pilots ca...",A situation where one side faces overwhelming ...,kaikki_wiktionary,dictionary,noun,"broadly, idiomatic",high,...,The phrase 'turkey shoot' is idiomatic because...,"تعبير ""turkey shoot"" تعبير اصطلاحي لأنه يصف بش...",This example clearly uses the phrase figurativ...,يستخدم هذا المثال العبارة بشكل مجازي لتوصيل مع...,pair_9d8101efff7b,paraphrase_305734ad2f7a,0,NaN,idiomatic,main_example
1,idiomx_e3e7571a5cef,turkey shoot,turkey shoot,"In the Battle of the Marianas, which pilots ca...",A situation where one side faces overwhelming ...,kaikki_wiktionary,dictionary,noun,"broadly, idiomatic",high,...,The phrase 'turkey shoot' is idiomatic because...,"تعبير ""turkey shoot"" تعبير اصطلاحي لأنه يصف بش...","Used literally, the phrase refers to an actual...",باستخدام حرفي، تشير العبارة إلى حدث صيد ديك رو...,pair_9d8101efff7b,paraphrase_305734ad2f7a,0,NaN,literal,main_example
2,idiomx_e3e7571a5cef,turkey shoot,turkey shoot,"In the Battle of the Marianas, which pilots ca...",A situation where one side faces overwhelming ...,kaikki_wiktionary,dictionary,noun,"broadly, idiomatic",high,...,The phrase 'turkey shoot' is idiomatic because...,"تعبير ""turkey shoot"" تعبير اصطلاحي لأنه يصف بش...",This narrative uses the phrase figuratively to...,يستخدم هذا السرد العبارة مجازيًا للدلالة على ه...,pair_dd862b5ab208,paraphrase_305734ad2f7a,0,NaN,idiomatic,main_example



Preview: rows sent for review -> none

Preview: corrected rows -> none

Preview: verification log -> none

Verification result summary:
- Final dataset rows      : 28
- Rows sent for review    : 0
- Rows corrected          : 0
- Verification log rows   : 0


### Verification Completed

The dataset has been reviewed and corrected.

- Suspicious rows have been fixed or confirmed
- A final clean dataset is now available

Next step: Export the final dataset.

---
### [8] Export final dataset

The final dataset is exported in two formats:

- CSV → human-readable and easy to inspect
- Parquet → optimized for machine learning pipelines

Priority is given to the fully verified dataset when available.

All outputs are saved under:

data/final/

In [20]:
# [8.1] Export final dataset

from pathlib import Path
import pandas as pd

print("Exporting final dataset...")

# ----------------------------
# Select best available dataset
# ----------------------------
if FINAL_VERIFIED_PARQUET_FILE.exists():
    source_file = FINAL_VERIFIED_PARQUET_FILE
    read_mode = "parquet"
elif FINAL_VERIFIED_CSV_FILE.exists():
    source_file = FINAL_VERIFIED_CSV_FILE
    read_mode = "csv"
elif VALIDATED_CSV_FILE.exists():
    source_file = VALIDATED_CSV_FILE
    read_mode = "csv"
else:
    raise FileNotFoundError(
        "No valid dataset found. Run validation/verification first."
    )

print("Source selected:", source_file)

# ----------------------------
# Load dataset
# ----------------------------
if read_mode == "parquet":
    final_df = pd.read_parquet(source_file)
else:
    final_df = pd.read_csv(source_file, low_memory=False)

# Optional helper column for downstream use
if "row_type" in final_df.columns and "is_adversarial" not in final_df.columns:
    final_df["is_adversarial"] = final_df["row_type"].astype(str).str.contains("adversarial", na=False)
    
# ----------------------------
# Ensure output folders exist
# ----------------------------
FINAL_EXPORT_CSV_FILE.parent.mkdir(parents=True, exist_ok=True)
FINAL_EXPORT_PARQUET_FILE.parent.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Save outputs
# ----------------------------
final_df.to_csv(FINAL_EXPORT_CSV_FILE, index=False, encoding="utf-8-sig")
final_df.to_parquet(FINAL_EXPORT_PARQUET_FILE, index=False)

print("\nFinal dataset exported successfully.")
print("CSV     :", FINAL_EXPORT_CSV_FILE)
print("Parquet :", FINAL_EXPORT_PARQUET_FILE)
print("Shape   :", final_df.shape)

# Save lightweight public sample
FINAL_SAMPLE_CSV_FILE = FINAL_DIR / "idiomx_final_sample_v2.csv"
FINAL_SAMPLE_PARQUET_FILE = FINAL_DIR / "idiomx_final_sample_v2.parquet"

sample_n = min(1000, len(final_df))
final_sample_df = final_df.sample(sample_n, random_state=42)

final_sample_df.to_csv(FINAL_SAMPLE_CSV_FILE, index=False, encoding="utf-8-sig")
final_sample_df.to_parquet(FINAL_SAMPLE_PARQUET_FILE, index=False)

print("Final sample CSV     :", FINAL_SAMPLE_CSV_FILE)
print("Final sample Parquet :", FINAL_SAMPLE_PARQUET_FILE)
print("Final sample shape   :", final_sample_df.shape)
# ----------------------------
# Optional preview
# ----------------------------
display(final_df.head(3))

Exporting final dataset...
Source selected: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\enriched\idiomx_enriched_final_sample_v2.parquet

Final dataset exported successfully.
CSV     : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\final\idiomx_final_sample_v2.csv
Parquet : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\final\idiomx_final_sample_v2.parquet
Shape   : (28, 48)
Final sample CSV     : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\final\idiomx_final_sample_v2.csv
Final sample Parquet : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\final\idiomx_final_sample_v2.parquet
Final sample shape   : (28, 48)


,idiom_id,idiom_canonical,idiom_surface,example,idiom_canonical_meaning,source,source_type,pos,tags,idiom_confidence,...,idiom_level_explanation_ar,explanation_en,explanation_ar,minimal_pair_id,paraphrase_group_id,is_adversarial_example,adversarial_type,expected_label,row_type,is_adversarial
0,idiomx_e3e7571a5cef,turkey shoot,turkey shoot,"In the Battle of the Marianas, which pilots ca...",A situation where one side faces overwhelming ...,kaikki_wiktionary,dictionary,noun,"broadly, idiomatic",high,...,"تعبير ""turkey shoot"" تعبير اصطلاحي لأنه يصف بش...",This example clearly uses the phrase figurativ...,يستخدم هذا المثال العبارة بشكل مجازي لتوصيل مع...,pair_9d8101efff7b,paraphrase_305734ad2f7a,0,None,idiomatic,main_example,False
1,idiomx_e3e7571a5cef,turkey shoot,turkey shoot,"In the Battle of the Marianas, which pilots ca...",A situation where one side faces overwhelming ...,kaikki_wiktionary,dictionary,noun,"broadly, idiomatic",high,...,"تعبير ""turkey shoot"" تعبير اصطلاحي لأنه يصف بش...","Used literally, the phrase refers to an actual...",باستخدام حرفي، تشير العبارة إلى حدث صيد ديك رو...,pair_9d8101efff7b,paraphrase_305734ad2f7a,0,None,literal,main_example,False
2,idiomx_e3e7571a5cef,turkey shoot,turkey shoot,"In the Battle of the Marianas, which pilots ca...",A situation where one side faces overwhelming ...,kaikki_wiktionary,dictionary,noun,"broadly, idiomatic",high,...,"تعبير ""turkey shoot"" تعبير اصطلاحي لأنه يصف بش...",This narrative uses the phrase figuratively to...,يستخدم هذا السرد العبارة مجازيًا للدلالة على ه...,pair_dd862b5ab208,paraphrase_305734ad2f7a,0,None,idiomatic,main_example,False


In [22]:
# [9.1] Final QA summary

print("Final pipeline QA summary")

# ----------------------------
# File existence checks
# ----------------------------
def check_files(title, paths):
    print(f"\n{title}")
    for p in paths:
        print(f" - {p} | exists = {p.exists()}")

check_files("Source files", [INPUT_PRE_FILE])

check_files("Intermediate outputs", [
    BATCH_JSONL_FILE,
    BATCH_INFO_FILE,
    RESULTS_JSONL_FILE,
    ENRICHED_CSV_FILE,
    VALIDATED_CSV_FILE,
    ISSUES_CSV_FILE,
    FINAL_VERIFIED_CSV_FILE,
])

check_files("Audit outputs", [
    REVIEW_ROWS_FILE,
    CORRECTED_ROWS_FILE,
    VERIFICATION_LOG_FILE,
])

check_files("Final outputs", [
    FINAL_EXPORT_CSV_FILE,
    FINAL_EXPORT_PARQUET_FILE,
])

# ----------------------------
# Dataset stats
# ----------------------------
print("\nFinal dataset shape:", final_df.shape)

def safe_value_counts(df, col):
    if col in df.columns:
        print(f"\n{col} distribution:")
        print(df[col].value_counts(dropna=False))

safe_value_counts(final_df, "validation_status")
safe_value_counts(final_df, "row_type")
safe_value_counts(final_df, "example_usage_label")

# ----------------------------
# Data integrity checks (NEW 🔥)
# ----------------------------
if "idiom_id" in final_df.columns:
    print("\nUnique idioms:", final_df["idiom_id"].nunique())

if "idiom_canonical" in final_df.columns:
    print("Unique canonical idioms:", final_df["idiom_canonical"].nunique())

# Check duplicates
dup_count = final_df.duplicated().sum()
print("\nDuplicate rows:", dup_count)

# Missing values overview
print("\nMissing values (top 10 columns):")
print(final_df.isna().sum().sort_values(ascending=False).head(10))

Final pipeline QA summary

Source files
 - C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\final\idiomx_pre_enrichment.parquet | exists = True

Intermediate outputs
 - C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\batches\idiomx_batch_sample_v2.jsonl | exists = True
 - C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\batches\idiomx_batch_info_sample_v2.json | exists = True
 - C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\results\idiomx_results_sample_v2.jsonl | exists = True
 - C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\enriched\idiomx_enriched_sample_v2.csv | exists = True
 - C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\enriched\idiomx_enriched_validated_sample_v2.csv | exists = True
 - C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_dataset_exp\data\enriched\idiomx_enriched_issues_sample_v2.csv | exists = True
 - C:\Users\ayman\Documents\

In [ ]:
# fallback export in case if llm not enabled 
import pandas as pd

print("Preparing final export / fallback dataset...")

source_file = None
final_target_csv = FINAL_EXPORT_CSV_FILE
final_target_parquet = FINAL_EXPORT_PARQUET_FILE

if FINAL_VERIFIED_CSV_FILE.exists():
    source_file = FINAL_VERIFIED_CSV_FILE
    print("Using FINAL_VERIFIED_CSV_FILE")
elif VALIDATED_CSV_FILE.exists():
    source_file = VALIDATED_CSV_FILE
    print("Using VALIDATED_CSV_FILE")
elif ENRICHED_CSV_FILE.exists():
    source_file = ENRICHED_CSV_FILE
    print("Using ENRICHED_CSV_FILE")
elif INPUT_PRE_FILE.exists():
    source_file = INPUT_PRE_FILE
    print("No enriched outputs found. Using pre-enrichment input as fallback stub.")
else:
    raise FileNotFoundError("No usable source file found for final export.")

if source_file.suffix.lower() == ".parquet":
    final_df = pd.read_parquet(source_file)
else:
    final_df = pd.read_csv(source_file, low_memory=False)

FINAL_DIR.mkdir(parents=True, exist_ok=True)

final_df.to_csv(final_target_csv, index=False, encoding="utf-8-sig")
final_df.to_parquet(final_target_parquet, index=False)

print("Export completed.")
print("Source:", source_file)
print("CSV:", final_target_csv)
print("Parquet:", final_target_parquet)
print("Shape:", final_df.shape)

### Verification Note

A heuristic validation stage identified a small subset of rows requiring further review. Since the validated dataset already achieved a high validity rate (96.35%), the main experiments in this notebook proceed using the validated dataset. Full manual or LLM-assisted correction can be applied later as an additional refinement stage.

## Final Dataset

The final dataset used in this notebook consists of **179,883 examples**, derived from the validated enrichment pipeline.

- Source: validated enriched dataset (96.35% valid)
- Format: CSV and Parquet
- Location:
  - `data/final/idiomx_final_v2.csv`
  - `data/final/idiomx_final_v2.parquet`

This dataset serves as the foundation for all Task 2 experiments.

### Notes

- A small subset of rows (~3.6%) was flagged for potential issues during validation.
- These rows were retained to preserve dataset diversity and scale.
- Additional verification can be applied as a future refinement step.

### Key Observations

The final IdiomX v2 dataset consists of **179,883 examples** spanning **12,678 unique idioms**, providing substantial lexical diversity for idiom understanding tasks.

#### Dataset Composition
- The dataset is dominated by **main examples (154K rows)**, complemented by **25K adversarial examples** designed to increase task difficulty and robustness.
- Adversarial examples account for approximately **14.3%** of the dataset.

#### Usage Label Distribution
- The dataset is relatively balanced between:
  - **Literal usage (~85K)**
  - **Idiomatic usage (~80K)**
- A smaller portion (**~13.5K**) is labeled as **borderline**, introducing ambiguity and increasing task complexity.

#### Data Quality
- Missing values in critical fields are minimal:
  - Only **350 rows (~0.19%)** missing canonical idioms.
- The `idiom_in_example` field is fully populated, ensuring complete contextual coverage.

#### Text Characteristics
- The average example length is **~73 characters**, with most examples ranging between **63–84 characters**.
- However, the dataset includes rare long-tail examples (up to **7,037 characters**), which may introduce noise or outliers.

#### Idiom Distribution
- Idioms are consistently represented across the dataset, with no single-instance idioms.
- Most idioms appear multiple times, supporting reliable training and evaluation.

#### Implications for Modeling
- The presence of adversarial examples and borderline cases makes the dataset particularly suitable for:
  - retrieval-based models
  - hybrid ranking systems
- The balanced label distribution ensures that models must learn both **literal vs idiomatic distinction** and **contextual semantic mapping**.

Overall, the dataset provides a **challenging and diverse benchmark** for idiom understanding and retrieval tasks.

In [1]:
print("\nNOTE:")
print("This notebook runs in SAMPLE mode.")
print("Full dataset enrichment results are provided in the dataset files and README.")


NOTE:
This notebook runs in SAMPLE mode.
Full dataset enrichment results are provided in the dataset files and README.


## Next notebook Final cleaning